# Notebook 03 — Persistent 3D Patch Extraction for RibFrac

This notebook extracts fixed-size 3D CT patches and multi-task targets from the persistent preprocessed RibFrac dataset produced by Notebook 02.

## Dataset
- Labelled cohort: 500 patients
- Training Part 1: 300
- Training Part 2: 120
- Labelled validation: 80
- Patient-level split: 350 train / 75 validation / 75 internal held-out test
- Split seed: 42

## Patch policy
- Patch size: `(64, 64, 64)`
- Stride: `(32, 32, 32)`
- Segmentation target: `0 = background`, `1 = valid fracture`
- Classification target: `0 = background`, `1 = displaced`, `2 = non-displaced`, `3 = buckle`, `4 = segmental`
- `label_code = -1` is ambiguous and is excluded.
- Positive patches contain at least one valid fracture voxel.
- Negative patches contain no valid fracture voxels.
- Negative sampling is capped at a 1:1 negative-to-positive ratio.
- Mixed-class ties are excluded because classification uses a single-label softmax target.
- No patch-level train/validation/test split is performed.

**Important:** Notebook 03 does not contact Zenodo and does not redo CT preprocessing. It consumes the `.npy` volumes generated by Notebook 02.

In [ ]:
# Cell 1 — Imports and reproducibility

import gc
import json
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Libraries imported successfully.")
print("Random seed:", SEED)

In [ ]:
# Cell 2 — Kaggle input and persistent patch-output configuration

# Published Notebook 02 dataset.
# Change only DATASET_ROOT if Kaggle displays a different mount path.
DATASET_ROOT = Path(
    "/kaggle/input/datasets/waboke/ribfrac-preprocessed-final"
)

PREPROCESSED_ROOT = DATASET_ROOT
IMAGE_DIR = PREPROCESSED_ROOT / "images"
LABEL_DIR = PREPROCESSED_ROOT / "labels"
METADATA_DIR = PREPROCESSED_ROOT / "metadata"

PATCH_ROOT = Path(
    "/kaggle/working/ribfrac-patch-dataset-final"
)
PATCH_METADATA_DIR = PATCH_ROOT / "metadata"
PATCH_STATUS_PATH = PATCH_METADATA_DIR / "processing_status.csv"

for path in [PATCH_ROOT, PATCH_METADATA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Input dataset:", DATASET_ROOT)
print("Image directory:", IMAGE_DIR)
print("Label directory:", LABEL_DIR)
print("Metadata directory:", METADATA_DIR)
print("Patch output:", PATCH_ROOT)

In [ ]:
# Cell 3 — Patch and target configuration

PATCH_SIZE = (64, 64, 64)
STRIDE = (32, 32, 32)

NEGATIVE_TO_POSITIVE_RATIO = 1.0
MIN_FRACTURE_VOXELS = 1
AMBIGUOUS_LABEL_CODE = -1

CLASS_NAMES = {
    0: "background",
    1: "displaced",
    2: "non-displaced",
    3: "buckle",
    4: "segmental",
}

LABEL_CODE_TO_CLASS = {
    1: 1,
    2: 2,
    3: 3,
    4: 4,
}

EXPECTED_SPLIT_COUNTS = {
    "train": 350,
    "validation": 75,
    "test": 75,
}

MAX_PATIENTS_PER_RUN = 1
RESUME_EXISTING = True

print("Patch size:", PATCH_SIZE)
print("Stride:", STRIDE)
print("Negative/positive ratio:", NEGATIVE_TO_POSITIVE_RATIO)
print("Classification classes:", CLASS_NAMES)
print("MAX_PATIENTS_PER_RUN:", MAX_PATIENTS_PER_RUN)

## Input validation

Notebook 02 stores one CT and one label volume per patient:

```text
images/RibFracXXX-image.npy
labels/RibFracXXX-label.npy
metadata/
    patient_source_registry_500.csv
    patient_split_500.csv
    combined_ribfrac_metadata.csv
    processing_status.csv
```

The patch extractor uses the existing patient registry and split. It does not recreate the split.

In [ ]:
# Cell 4 — Verify persistent Notebook 02 dataset

for path in [IMAGE_DIR, LABEL_DIR, METADATA_DIR]:
    assert path.exists(), f"Required directory not found: {path}"

SPLIT_PATH = METADATA_DIR / "patient_split_500.csv"
REGISTRY_PATH = METADATA_DIR / "patient_source_registry_500.csv"
PROCESSING_STATUS_PATH = METADATA_DIR / "processing_status.csv"

assert SPLIT_PATH.exists(), f"Missing patient split: {SPLIT_PATH}"
assert REGISTRY_PATH.exists(), f"Missing patient registry: {REGISTRY_PATH}"

patient_split = pd.read_csv(SPLIT_PATH)
patient_registry = pd.read_csv(REGISTRY_PATH)

print("Persistent dataset verification: PASS")
print("Patients in split:", len(patient_split))
print("Patients in registry:", len(patient_registry))
print("\nSplit counts:")
print(patient_split["split"].value_counts())

In [ ]:
# Cell 5 — Verify 500-patient split

def find_column(df, candidates):
    lower_map = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    return None

split_patient_col = find_column(patient_split, ["public_id", "patient_id", "id"])
registry_patient_col = find_column(patient_registry, ["public_id", "patient_id", "id"])

assert split_patient_col is not None, patient_split.columns.tolist()
assert registry_patient_col is not None, patient_registry.columns.tolist()

if split_patient_col != "public_id":
    patient_split = patient_split.rename(columns={split_patient_col: "public_id"})
if registry_patient_col != "public_id":
    patient_registry = patient_registry.rename(columns={registry_patient_col: "public_id"})

patient_split["public_id"] = patient_split["public_id"].astype(str)
patient_registry["public_id"] = patient_registry["public_id"].astype(str)

assert len(patient_split) == 500
assert patient_split["public_id"].nunique() == 500
assert set(patient_split["split"].unique()) == {"train", "validation", "test"}

actual_counts = patient_split["split"].value_counts().to_dict()
assert actual_counts == EXPECTED_SPLIT_COUNTS, actual_counts
assert set(patient_split["public_id"]) == set(patient_registry["public_id"])

train_patients = sorted(patient_split.loc[patient_split["split"]=="train", "public_id"])
validation_patients = sorted(patient_split.loc[patient_split["split"]=="validation", "public_id"])
test_patients = sorted(patient_split.loc[patient_split["split"]=="test", "public_id"])

print("Patient-level split verification: PASS")
print("Train:", len(train_patients))
print("Validation:", len(validation_patients))
print("Internal test:", len(test_patients))

In [ ]:
# Cell 6 — Verify Notebook 02 processing status

if PROCESSING_STATUS_PATH.exists():
    processing_status = pd.read_csv(PROCESSING_STATUS_PATH)
    processing_status["public_id"] = processing_status["public_id"].astype(str)
    completed_ids = set(
        processing_status.loc[
            processing_status["status"] == "completed",
            "public_id"
        ]
    )
    print("Notebook 02 completed patients:", len(completed_ids))
    print(processing_status["status"].value_counts())
else:
    processing_status = pd.DataFrame()
    completed_ids = set()
    print("Warning: processing_status.csv was not found.")

## Annotation mapping

RibFrac label volumes contain annotation/instance IDs, not the final fracture classes.

The `label_code` in the patient metadata determines the fracture class:

- `-1` → ambiguous / ignore
- `1` → displaced
- `2` → non-displaced
- `3` → buckle
- `4` → segmental

Background is the voxel value `0` in the annotation volume.

Therefore, annotation ID `0` must not be confused with metadata `label_id = 0`.

In [ ]:
# Cell 7 — Load combined annotation metadata

COMBINED_METADATA_PATH = METADATA_DIR / "combined_ribfrac_metadata.csv"
assert COMBINED_METADATA_PATH.exists(), f"Missing: {COMBINED_METADATA_PATH}"

combined_metadata = pd.read_csv(COMBINED_METADATA_PATH)

def first_existing_column(df, candidates):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    return None

patient_col = first_existing_column(combined_metadata, ["public_id", "patient_id", "id"])
label_id_col = first_existing_column(combined_metadata, ["label_id"])
label_code_col = first_existing_column(combined_metadata, ["label_code"])

assert patient_col is not None
assert label_id_col is not None
assert label_code_col is not None

combined_metadata = combined_metadata.rename(columns={
    patient_col: "public_id",
    label_id_col: "label_id",
    label_code_col: "label_code",
})

combined_metadata["public_id"] = combined_metadata["public_id"].astype(str)
combined_metadata["label_id"] = pd.to_numeric(combined_metadata["label_id"], errors="coerce")
combined_metadata["label_code"] = pd.to_numeric(combined_metadata["label_code"], errors="coerce")

print("Annotation metadata records:", len(combined_metadata))
print("Metadata mapping loaded successfully.")

In [ ]:
# Cell 8 — Build per-patient annotation-ID mappings

patient_annotation_maps = {}

for patient_id, group in combined_metadata.groupby("public_id"):
    mapping = {}

    for _, row in group.iterrows():
        if pd.isna(row["label_id"]):
            continue

        annotation_id = int(row["label_id"])
        class_code = (
            AMBIGUOUS_LABEL_CODE
            if pd.isna(row["label_code"])
            else int(row["label_code"])
        )
        mapping[annotation_id] = class_code

    mapping[0] = 0
    patient_annotation_maps[str(patient_id)] = mapping

print("Patients with annotation metadata:", len(patient_annotation_maps))

example_patient = sorted(patient_annotation_maps.keys())[0]
print("Example patient:", example_patient)
print("Example mapping:", patient_annotation_maps[example_patient])

In [ ]:
# Cell 9 — Persistent patch storage helpers

def patient_patch_dir(split_name, patient_id):
    return PATCH_ROOT / split_name / str(patient_id)

def patient_status_path(split_name, patient_id):
    return patient_patch_dir(split_name, patient_id) / "patient_patch_status.json"

def patient_complete(split_name, patient_id):
    status_path = patient_status_path(split_name, patient_id)

    if not status_path.exists():
        return False

    try:
        with open(status_path, "r", encoding="utf-8") as f:
            status = json.load(f)
        return status.get("status") == "completed"
    except Exception:
        return False

print("Patch storage helpers ready.")

## Patch extraction logic

For each patient:

1. Load the persistent 1-mm CT and label arrays.
2. Generate `(64,64,64)` windows using stride `(32,32,32)`.
3. Convert annotation IDs to fracture classes using patient metadata.
4. Exclude patches containing ambiguous annotations.
5. Create a binary segmentation mask.
6. Assign a single five-class classification label to positive patches.
7. Exclude ties between multiple valid fracture classes.
8. Sample negative patches at a maximum 1:1 ratio.
9. Save the patient's patches and manifest.
10. Mark the patient as completed.

In [ ]:
# Cell 10 — Generate patch start coordinates

def generate_patch_starts(shape, patch_size, stride):
    starts = []

    for d in range(0, shape[0] - patch_size[0] + 1, stride[0]):
        for h in range(0, shape[1] - patch_size[1] + 1, stride[1]):
            for w in range(0, shape[2] - patch_size[2] + 1, stride[2]):
                starts.append((d, h, w))

    return starts

test_starts = generate_patch_starts((128, 128, 128), PATCH_SIZE, STRIDE)
print("128³ test volume patch count:", len(test_starts))
assert len(test_starts) > 0

In [ ]:
# Cell 11 — Convert annotation IDs and generate targets

def build_class_volume(label_volume, annotation_mapping):
    class_volume = np.zeros(label_volume.shape, dtype=np.int8)

    for annotation_id in np.unique(label_volume):
        annotation_id = int(annotation_id)

        if annotation_id == 0:
            continue

        class_code = annotation_mapping.get(
            annotation_id,
            AMBIGUOUS_LABEL_CODE
        )

        class_volume[label_volume == annotation_id] = class_code

    return class_volume


def make_targets_from_class_patch(class_patch):
    ambiguous_mask = class_patch == AMBIGUOUS_LABEL_CODE

    if np.any(ambiguous_mask):
        return None, None, False, "ambiguous"

    valid_fracture = np.isin(class_patch, [1, 2, 3, 4])

    if not np.any(valid_fracture):
        segmentation = np.zeros((*PATCH_SIZE, 1), dtype=np.uint8)
        return segmentation, 0, False, "negative"

    segmentation = valid_fracture.astype(np.uint8)[..., None]

    class_counts = Counter(class_patch[valid_fracture].tolist())
    max_count = max(class_counts.values())

    dominant_classes = [
        cls for cls, count in class_counts.items()
        if count == max_count
    ]

    if len(dominant_classes) != 1:
        return None, None, False, "mixed_class_tie"

    return (
        segmentation,
        int(dominant_classes[0]),
        True,
        "positive",
    )

In [ ]:
# Cell 12 — Extract and save one patient

def process_patient(patient_id, split_name):

    patient_id = str(patient_id)
    output_dir = patient_patch_dir(split_name, patient_id)
    output_dir.mkdir(parents=True, exist_ok=True)

    image_path = IMAGE_DIR / f"{patient_id}-image.npy"
    label_path = LABEL_DIR / f"{patient_id}-label.npy"

    if not image_path.exists():
        raise FileNotFoundError(f"CT not found: {image_path}")
    if not label_path.exists():
        raise FileNotFoundError(f"Label not found: {label_path}")
    if patient_id not in patient_annotation_maps:
        raise KeyError(f"No annotation metadata found for {patient_id}")

    ct = np.load(image_path, mmap_mode="r")
    label = np.load(label_path, mmap_mode="r")

    assert ct.ndim == 3
    assert label.ndim == 3
    assert ct.shape == label.shape

    print("Volume shape:", ct.shape)

    label_array = np.asarray(label, dtype=np.int16)
    class_volume = build_class_volume(
        label_array,
        patient_annotation_maps[patient_id]
    )

    starts = generate_patch_starts(
        ct.shape,
        PATCH_SIZE,
        STRIDE
    )

    positive_images = []
    positive_segmentations = []
    positive_classes = []
    positive_starts = []

    negative_images = []
    negative_segmentations = []
    negative_classes = []
    negative_starts = []

    skipped_ambiguous = 0
    skipped_ties = 0

    for d, h, w in starts:

        ct_patch = np.asarray(
            ct[
                d:d+PATCH_SIZE[0],
                h:h+PATCH_SIZE[1],
                w:w+PATCH_SIZE[2]
            ],
            dtype=np.float32
        )

        class_patch = class_volume[
            d:d+PATCH_SIZE[0],
            h:h+PATCH_SIZE[1],
            w:w+PATCH_SIZE[2]
        ]

        segmentation, classification, positive, reason = (
            make_targets_from_class_patch(class_patch)
        )

        if reason == "ambiguous":
            skipped_ambiguous += 1
            continue

        if reason == "mixed_class_tie":
            skipped_ties += 1
            continue

        ct_patch = ct_patch[..., None]

        if positive:
            positive_images.append(ct_patch)
            positive_segmentations.append(segmentation)
            positive_classes.append(classification)
            positive_starts.append((d, h, w))
        else:
            negative_images.append(ct_patch)
            negative_segmentations.append(segmentation)
            negative_classes.append(classification)
            negative_starts.append((d, h, w))

    # Deterministic 1:1 negative sampling.
    rng = np.random.default_rng(
        SEED + sum(ord(c) for c in patient_id)
    )

    max_negative = int(
        len(positive_images) * NEGATIVE_TO_POSITIVE_RATIO
    )

    if len(negative_images) > max_negative:
        selected = rng.choice(
            len(negative_images),
            size=max_negative,
            replace=False
        )

        negative_images = [negative_images[i] for i in selected]
        negative_segmentations = [negative_segmentations[i] for i in selected]
        negative_classes = [negative_classes[i] for i in selected]
        negative_starts = [negative_starts[i] for i in selected]

    images = positive_images + negative_images
    segmentations = positive_segmentations + negative_segmentations
    classifications = positive_classes + negative_classes
    starts_final = positive_starts + negative_starts

    if len(images) == 0:
        raise RuntimeError(
            f"No usable patches generated for {patient_id}. "
            f"Ambiguous={skipped_ambiguous}, ties={skipped_ties}"
        )

    images = np.stack(images).astype(np.float32)
    segmentations = np.stack(segmentations).astype(np.uint8)
    classifications = np.asarray(classifications, dtype=np.int8)

    np.save(output_dir / "images.npy", images)
    np.save(output_dir / "segmentation.npy", segmentations)
    np.save(output_dir / "classification.npy", classifications)

    manifest = pd.DataFrame({
        "patient_id": [patient_id] * len(starts_final),
        "split": [split_name] * len(starts_final),
        "patch_index": np.arange(len(starts_final)),
        "start_d": [s[0] for s in starts_final],
        "start_h": [s[1] for s in starts_final],
        "start_w": [s[2] for s in starts_final],
        "classification": classifications,
        "positive": [
            1 if i < len(positive_images) else 0
            for i in range(len(images))
        ],
    })

    manifest.to_csv(
        output_dir / "patch_manifest.csv",
        index=False
    )

    status = {
        "status": "completed",
        "patient_id": patient_id,
        "split": split_name,
        "volume_shape": list(ct.shape),
        "total_candidate_patches": len(starts),
        "positive_patches": len(positive_images),
        "negative_patches": len(negative_images),
        "skipped_ambiguous": skipped_ambiguous,
        "skipped_mixed_class_ties": skipped_ties,
        "saved_patches": len(images),
        "image_shape": list(images.shape),
        "segmentation_shape": list(segmentations.shape),
        "classification_shape": list(classifications.shape),
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }

    with open(
        patient_status_path(split_name, patient_id),
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(status, f, indent=2)

    del label_array, class_volume
    del images, segmentations, classifications
    gc.collect()

    return status

In [ ]:
# Cell 13 — Resumable patient-by-patient extraction

registry_to_process = (
    patient_split
    .sort_values(["split", "public_id"])
    .reset_index(drop=True)
)

processed_this_run = 0
run_results = []

for _, row in registry_to_process.iterrows():

    patient_id = str(row["public_id"])
    split_name = str(row["split"])

    if RESUME_EXISTING and patient_complete(split_name, patient_id):
        print(f"{patient_id}: already completed — SKIP")
        continue

    if (
        MAX_PATIENTS_PER_RUN is not None
        and processed_this_run >= MAX_PATIENTS_PER_RUN
    ):
        print(
            f"Reached MAX_PATIENTS_PER_RUN="
            f"{MAX_PATIENTS_PER_RUN}."
        )
        break

    print("\n" + "=" * 80)
    print(f"PROCESSING {patient_id} | split={split_name}")
    print("=" * 80)

    try:
        result = process_patient(patient_id, split_name)
        run_results.append(result)
        processed_this_run += 1

        print(
            f"{patient_id}: COMPLETE | "
            f"patches={result['saved_patches']}"
        )

    except Exception as exc:

        print(
            f"{patient_id}: FAILED → {repr(exc)}"
        )

        error_status = {
            "status": "failed",
            "patient_id": patient_id,
            "split": split_name,
            "error": repr(exc),
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        }

        output_dir = patient_patch_dir(split_name, patient_id)
        output_dir.mkdir(parents=True, exist_ok=True)

        with open(
            patient_status_path(split_name, patient_id),
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(error_status, f, indent=2)

        gc.collect()

print()
print("=" * 80)
print("Patients completed this run:", processed_this_run)
print("=" * 80)

In [ ]:
# Cell 14 — Build global patch manifest

manifest_frames = []

for split_name in ["train", "validation", "test"]:

    split_dir = PATCH_ROOT / split_name

    if not split_dir.exists():
        continue

    for patient_dir in sorted(split_dir.iterdir()):

        if not patient_dir.is_dir():
            continue

        manifest_path = patient_dir / "patch_manifest.csv"

        if manifest_path.exists():
            manifest_frames.append(
                pd.read_csv(manifest_path)
            )

if manifest_frames:
    global_manifest = pd.concat(
        manifest_frames,
        ignore_index=True
    )
else:
    global_manifest = pd.DataFrame(columns=[
        "patient_id", "split", "patch_index",
        "start_d", "start_h", "start_w",
        "classification", "positive"
    ])

GLOBAL_MANIFEST_PATH = PATCH_METADATA_DIR / "patch_manifest.csv"

global_manifest.to_csv(
    GLOBAL_MANIFEST_PATH,
    index=False
)

print("Global manifest saved:", GLOBAL_MANIFEST_PATH)
print("Total saved patches:", len(global_manifest))

In [ ]:
# Cell 15 — Patient patch summary and class distribution

summary_rows = []

for split_name in ["train", "validation", "test"]:

    split_dir = PATCH_ROOT / split_name

    if not split_dir.exists():
        continue

    for patient_dir in sorted(split_dir.iterdir()):

        if not patient_dir.is_dir():
            continue

        status_path = patient_dir / "patient_patch_status.json"

        if not status_path.exists():
            continue

        with open(status_path, "r", encoding="utf-8") as f:
            summary_rows.append(json.load(f))

patient_summary = pd.DataFrame(summary_rows)

PATIENT_SUMMARY_PATH = (
    PATCH_METADATA_DIR / "patient_patch_summary.csv"
)

patient_summary.to_csv(
    PATIENT_SUMMARY_PATH,
    index=False
)

if len(global_manifest):
    class_distribution = (
        global_manifest
        .groupby(["split", "classification"])
        .size()
        .reset_index(name="patch_count")
    )
else:
    class_distribution = pd.DataFrame(
        columns=["split", "classification", "patch_count"]
    )

CLASS_DISTRIBUTION_PATH = (
    PATCH_METADATA_DIR / "class_distribution.csv"
)

class_distribution.to_csv(
    CLASS_DISTRIBUTION_PATH,
    index=False
)

print("Patient summary:", PATIENT_SUMMARY_PATH)
print("Class distribution:", CLASS_DISTRIBUTION_PATH)
print("\nClass distribution:")
print(class_distribution)

In [ ]:
# Cell 16 — Save extraction configuration

config = {
    "notebook": "03_Patch_Extraction_RibFrac_Final_Persistent_500Patients",
    "seed": SEED,
    "patch_size": list(PATCH_SIZE),
    "stride": list(STRIDE),
    "negative_to_positive_ratio": NEGATIVE_TO_POSITIVE_RATIO,
    "minimum_fracture_voxels": MIN_FRACTURE_VOXELS,
    "ambiguous_label_code": AMBIGUOUS_LABEL_CODE,
    "classification_classes": CLASS_NAMES,
    "label_code_to_class": LABEL_CODE_TO_CLASS,
    "patient_counts": EXPECTED_SPLIT_COUNTS,
    "patient_level_split": True,
    "patch_level_split": False,
    "official_ribfrac_test_ground_truth_used": False,
    "internal_test_is_labelled_holdout": True,
    "input_dataset": str(DATASET_ROOT),
}

CONFIG_PATH = (
    PATCH_METADATA_DIR / "patch_extraction_config.json"
)

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Configuration saved:", CONFIG_PATH)

In [ ]:
# Cell 17 — Validate saved patient patch files

validation_errors = []

for _, row in patient_summary.iterrows():

    if row.get("status") != "completed":
        continue

    patient_id = str(row["patient_id"])
    split_name = str(row["split"])
    patient_dir = patient_patch_dir(split_name, patient_id)

    image_path = patient_dir / "images.npy"
    segmentation_path = patient_dir / "segmentation.npy"
    classification_path = patient_dir / "classification.npy"
    manifest_path = patient_dir / "patch_manifest.csv"

    required = [
        image_path,
        segmentation_path,
        classification_path,
        manifest_path,
    ]

    for path in required:
        if not path.exists():
            validation_errors.append(f"Missing: {path}")

    if not all(path.exists() for path in required):
        continue

    images = np.load(image_path, mmap_mode="r")
    segmentations = np.load(segmentation_path, mmap_mode="r")
    classifications = np.load(classification_path, mmap_mode="r")
    manifest = pd.read_csv(manifest_path)

    if images.ndim != 5:
        validation_errors.append(
            f"{patient_id}: images ndim={images.ndim}"
        )

    if images.shape[1:] != (64, 64, 64, 1):
        validation_errors.append(
            f"{patient_id}: bad image shape {images.shape}"
        )

    if segmentations.shape[1:] != (64, 64, 64, 1):
        validation_errors.append(
            f"{patient_id}: bad segmentation shape {segmentations.shape}"
        )

    if len(images) != len(segmentations):
        validation_errors.append(
            f"{patient_id}: image/segmentation count mismatch"
        )

    if len(images) != len(classifications):
        validation_errors.append(
            f"{patient_id}: image/classification count mismatch"
        )

    if len(images) != len(manifest):
        validation_errors.append(
            f"{patient_id}: manifest count mismatch"
        )

    if not set(np.unique(segmentations).tolist()).issubset({0, 1}):
        validation_errors.append(
            f"{patient_id}: invalid segmentation values"
        )

    if not set(np.unique(classifications).tolist()).issubset(
        {0, 1, 2, 3, 4}
    ):
        validation_errors.append(
            f"{patient_id}: invalid classification values"
        )

    del images, segmentations, classifications
    gc.collect()

if validation_errors:
    print("VALIDATION FAILED")
    for error in validation_errors[:50]:
        print(error)
    raise AssertionError(
        f"{len(validation_errors)} validation errors found."
    )

print("Saved patch-file validation: PASS")

In [ ]:
# Cell 18 — Final Notebook 03 report

print("=" * 80)
print("NOTEBOOK 03 — PERSISTENT PATCH EXTRACTION REPORT")
print("=" * 80)

completed_patients = (
    patient_summary.loc[
        patient_summary["status"] == "completed"
    ]
    if len(patient_summary)
    else pd.DataFrame()
)

print("\nCompleted patients:", len(completed_patients))

if len(patient_summary):
    print("\nPatient status:")
    print(patient_summary["status"].value_counts())

if len(global_manifest):
    print("\nPatch counts by split:")
    print(global_manifest["split"].value_counts())

    print("\nClassification distribution:")
    print(
        global_manifest["classification"]
        .value_counts()
        .sort_index()
    )

print("\nPatch output:", PATCH_ROOT)

total_size_gb = (
    sum(
        p.stat().st_size
        for p in PATCH_ROOT.rglob("*")
        if p.is_file()
    )
    / (1024 ** 3)
)

print("Patch dataset size:", f"{total_size_gb:.3f} GB")

# Notebook 03 completion criteria

Notebook 03 is ready for Notebook 04/05 when:

1. The persistent Notebook 02 dataset is successfully mounted.
2. The 500-patient registry is verified.
3. The 350/75/75 patient split is verified.
4. Annotation IDs are mapped using RibFrac metadata.
5. Ambiguous annotations (`label_code = -1`) are excluded.
6. CT patches have shape `(64,64,64,1)`.
7. Segmentation targets contain only `{0,1}`.
8. Classification targets contain only `{0,1,2,3,4}`.
9. Positive patches contain valid fracture voxels.
10. Negative patches contain no valid fracture voxels.
11. Mixed-class ties are excluded.
12. CT, segmentation, and classification patch counts agree.
13. Patch manifests can be reloaded.
14. Patient-level split integrity is preserved.
15. No patch-level random splitting is performed.
16. Extraction is performed patient-by-patient and can resume.

## First execution

Keep:

```python
MAX_PATIENTS_PER_RUN = 1
RESUME_EXISTING = True
```

Run the notebook through Cell 18 and verify the first patient before increasing the batch size.